# Notebook 01 — Why Statistics for Biology

**Module:** 03 — Statistics and Probability  
**Notebook:** 01 of 18  
**Prerequisites:** Module 01 complete  
**Time estimate:** 45 minutes

---
## Step 1 — Motivation

Statistics is the most-tested non-coding skill in Indian research associate and
bioinformatics engineer interviews — more than any specific tool, more than any
programming technique. Questions like 'what is a t-test', 'what is FDR', and
'you have 20,000 p-values — what do you do' appear cold, without scaffolding.

This module exists to answer those questions rigorously, from first principles,
with working code — so that when the question comes, the answer is reflexive,
not assembled from half-remembered definitions.

---
## Step 2 — Intuition

Two types of questions in biology:

**Descriptive:** What is the mean expression of gene X across 100 samples?

**Inferential:** Is the expression of gene X genuinely different between cancer and
normal tissue, or could the observed difference be random sampling noise?

Descriptive statistics summarises data. Inferential statistics quantifies uncertainty —
it lets you make statements about populations from samples, with known error rates.

---
## Step 3 — Biological Background

**Why every biological measurement is noisy:**

- **Biological variability:** cells of the same type differ; individuals in a population
  differ; even genetically identical organisms differ due to stochastic gene expression
- **Technical variability:** sequencing errors, PCR amplification bias, batch effects,
  pipetting variation, RNA degradation
- **Sampling variability:** we can never measure all cells in a tumour, all individuals
  in a species, or all reads in a genome — we always work from samples

Statistics is the machinery for distinguishing signal from noise when both are always present.

**Where statistics appears in computational biology:**

| Task | Statistical method |
|------|-------------------|
| Differential expression testing | t-test, Wilcoxon, DESeq2 (negative binomial) |
| Genome-wide association studies | Logistic regression, multiple testing correction |
| ChIP-seq peak calling | Poisson / negative binomial models |
| Phylogenetic inference | Maximum likelihood, Bayesian posterior |
| Single-cell clustering | Statistical models for overdispersed count data |
| Sample quality control | Outlier detection (z-score, IQR rule) |

---
## Step 4 — Mathematical Explanation

**The language of statistics:**

- **Population:** the complete set you care about (all human genes, all cells of a type)
- **Sample:** what you actually measured (the 30 RNA-seq samples in your experiment)
- **Parameter:** a true property of the population (mean μ, variance σ²) — unknown
- **Statistic:** a quantity computed from the sample (sample mean $\bar{x}$, sample
  variance $s^2$) — an estimate of the parameter
- **Estimator:** the rule you use to compute a statistic from data
- **Sampling distribution:** the distribution of a statistic across many repeated samples
  — this is what lets you compute p-values and confidence intervals

---
## Step 5 — Computational Explanation

**Module 03 dependency map:**

```
NB01 (this) — why and orientation
├── NB02 — descriptive stats          → always needed first
│   └── NB03 — probability            → foundation for NB04–NB05
│       └── NB04 — Normal / CLT       → used in NB05–NB06
│           └── NB05 — hypothesis testing fundamentals
│               ├── NB06 — t-tests     → Track A core
│               ├── NB07 — ANOVA
│               └── NB08 — non-parametric
├── NB09 — correlation
├── NB10 — p-value misinterpretations  → critical thinking
│   └── NB11 — multiple testing        → Track A core, genomics standard
│       └── NB12 — power analysis
├── NB13 — Bayesian intuition
├── NB14 — resampling (bootstrap)
├── NB15 — regression
├── NB16 — expression data end-to-end
├── NB17 — mini project (portfolio)
└── NB18 — assessment + mock interview
```

---
## Step 6 — Python Implementation

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.stats as stats

In [ ]:
# Cell 6.1 — The noise problem: simulating what variability looks like in expression data
rng = np.random.default_rng(42)

# True expression (population mean) for a gene
mu_control = 5.0
mu_treatment = 6.5  # 30% increase
sigma = 1.5  # biological + technical noise

# Simulate small experiment (n=5 per group)
n_small = 5
ctrl_small = rng.normal(mu_control, sigma, n_small)
trt_small = rng.normal(mu_treatment, sigma, n_small)

# Simulate larger experiment (n=30 per group)
n_large = 30
ctrl_large = rng.normal(mu_control, sigma, n_large)
trt_large = rng.normal(mu_treatment, sigma, n_large)

print("Small experiment (n=5):")
print(f"  Control:   mean={ctrl_small.mean():.2f}  (true: {mu_control})")
print(f"  Treatment: mean={trt_small.mean():.2f}  (true: {mu_treatment})")
print(f"  Overlap obvious? Hard to tell with n=5")

print("\nLarge experiment (n=30):")
print(f"  Control:   mean={ctrl_large.mean():.2f}  (true: {mu_control})")
print(f"  Treatment: mean={trt_large.mean():.2f}  (true: {mu_treatment})")
print(f"  More reliable — why? The CLT (NB04) explains this.")

In [ ]:
# Cell 6.2 — The multiple testing reality in genomics
# If we test 20,000 genes at alpha=0.05, how many false positives do we expect?

n_genes = 20_000
alpha = 0.05
expected_false_positives = n_genes * alpha

print(f"Genes tested: {n_genes:,}")
print(f"Significance threshold: α = {alpha}")
print(f"Expected false positives (if all genes are null): {expected_false_positives:.0f}")
print(f"\nThis is the multiple testing problem — Module 03's most Track-A-critical topic.")
print(f"Solution: FDR correction (NB11). This is what 'q-value' means.")

---
## Step 7 — Visualization

In [ ]:
# Cell 7.1 — Module overview: the four core statistical tasks in genomics
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Panel 1: Noise in small vs. large samples
ax = axes[0]
ax.boxplot([ctrl_small, trt_small], positions=[1, 2], widths=0.4,
           patch_artist=True,
           boxprops=dict(facecolor="steelblue", alpha=0.6))
ax.boxplot([ctrl_large, trt_large], positions=[3.5, 4.5], widths=0.4,
           patch_artist=True,
           boxprops=dict(facecolor="tomato", alpha=0.6))
ax.axhline(mu_control, color="steelblue", ls=":", lw=1)
ax.axhline(mu_treatment, color="tomato", ls=":", lw=1)
ax.set_xticks([1, 2, 3.5, 4.5])
ax.set_xticklabels(["Ctrl\nn=5", "Trt\nn=5", "Ctrl\nn=30", "Trt\nn=30"], fontsize=8)
ax.set_ylabel("Expression (log2)"); ax.set_title("Sample size and noise (NB02, NB06)")

# Panel 2: P-value distribution under null (uniform) vs. signal (spike near 0)
ax = axes[1]
rng2 = np.random.default_rng(1)
null_pvals = rng2.uniform(0, 1, 19000)  # 95% null genes
signal_pvals = rng2.beta(0.5, 10, 1000)  # 5% real DE genes, small p-values
all_pvals = np.concatenate([null_pvals, signal_pvals])
ax.hist(all_pvals, bins=40, color="steelblue", edgecolor="white", density=True)
ax.axhline(1.0, color="gray", ls="--", lw=1, label="Uniform null")
ax.set_xlabel("p-value"); ax.set_ylabel("Density")
ax.set_title("P-value histogram (NB10, NB11)")
ax.legend(fontsize=8)

# Panel 3: Volcano plot preview
ax = axes[2]
rng3 = np.random.default_rng(2)
n = 500
log2fc = rng3.normal(0, 1, n)
neg_log10p = np.abs(rng3.normal(0, 1.5, n)) + rng3.exponential(0.5, n)
sig = (np.abs(log2fc) > 1) & (neg_log10p > 2)
ax.scatter(log2fc[~sig], neg_log10p[~sig], s=8, color="gray", alpha=0.4)
ax.scatter(log2fc[sig], neg_log10p[sig], s=12, color="tomato", alpha=0.8)
ax.axhline(2, color="black", ls="--", lw=0.8)
ax.axvline(-1, color="black", ls="--", lw=0.8)
ax.axvline(1, color="black", ls="--", lw=0.8)
ax.set_xlabel("log₂ fold change"); ax.set_ylabel("-log₁₀(FDR p-value)")
ax.set_title("Volcano plot preview (NB16, NB17)")

plt.suptitle("Module 03 preview — statistics at work in genomics", fontsize=11)
plt.tight_layout()
plt.show()

---
## Step 8 — Exercises

1. In your own words (no statistics jargon): what is the difference between a population
   parameter and a sample statistic? Give one biological example of each.
2. In Cell 6.2: if only 500 of the 20,000 genes are truly differentially expressed,
   and we test all 20,000 at α=0.05 without correction, what fraction of our 'significant'
   results will be false positives? Compute this.
3. Looking at the volcano plot (Cell 7.1): what do the dashed lines represent? What is
   plotted on each axis and why -log₁₀(p)?
4. Why does biological variability make statistics necessary? Write two sentences.

---
## Quiz — Active Recall

1. Name three sources of noise in an RNA-seq experiment.
2. What is the multiple testing problem? Why does it matter in genomics?
3. Distinguish descriptive from inferential statistics in one sentence each.
4. What is a sampling distribution? Why does it enable p-values?
5. Name four places in computational biology where statistics appears.

---
## Reflection

**Date completed:** ____________________

> *[Which of the four statistical tasks in the module overview figure is most unfamiliar? That should get extra attention in the corresponding notebook.]*

---
*Next: `02_descriptive_statistics_distributions.ipynb`*